In [105]:
# For later: RAD - https://github.com/ubc-systopia/dsn-2022-rad-artifact/tree/main

import pandas as pd
import glob
import os

data_paths = ["../data/rad/known_procedures/benign","../data/rad/known_procedures/anomaly", "../data/rad/unknown_procedures"]

files = []
for data_path in data_paths:
    files += sorted(glob.glob(os.path.join(data_path, "*.csv")))


# files = sorted(glob.glob(os.path.join(data_paths[0], "*.csv")))

# Load and concatenate
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [106]:
df.head()

,Timestamp,Module,Method_Name,Arguments,Responses,Exceptions,id,Execution Time (Sec),Arrival_Time,Departure_Time
0,2021:10:12:13:22:23.845243,C9,_init_,ftdi: None,NaN,NaN,NaN,NaN,NaN,NaN
1,2021:10:12:13:22:24.499940,C9,PING,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021:10:12:13:22:24.799885,C9,BIAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021:10:12:13:22:25.072611,C9,SPED,"velocity: 20000, acceleration: 20000",NaN,NaN,NaN,NaN,NaN,NaN
4,2021:10:12:13:22:25.453128,C9,BIAS,bias: 0,NaN,NaN,NaN,NaN,NaN,NaN


These data files contain lots of data that is not related to the robot, including the status of other machines and unrelated events. We will ignore the extra stuff.

In [107]:
df.drop(columns=['Module', 'Responses', "Exceptions", 'Arrival_Time', 'Departure_Time', 'Execution Time (Sec)', 'id'], inplace=True)
df.head()

,Timestamp,Method_Name,Arguments
0,2021:10:12:13:22:23.845243,_init_,ftdi: None
1,2021:10:12:13:22:24.499940,PING,NaN
2,2021:10:12:13:22:24.799885,BIAS,NaN
3,2021:10:12:13:22:25.072611,SPED,"velocity: 20000, acceleration: 20000"
4,2021:10:12:13:22:25.453128,BIAS,bias: 0


In [108]:
df = df[df["Method_Name"].str.contains("ARM", case=False, na=False)]
df = df[~df["Arguments"].str.contains("velocity", case=False, na=False)]
df.drop(columns=['Method_Name'], inplace=True)
df.reset_index(drop=True, inplace=True)

df

,Timestamp,Arguments
0,2021:10:12:14:35:31.590194,"X: 159525, Y: 182500, Z: 187000, gripper: 1089..."
1,2021:10:12:14:35:32.141240,"X: 160151, Y: 182500, Z: 187000, gripper: 1089..."
2,2021:10:12:14:35:32.717996,"X: 160776, Y: 182500, Z: 187000, gripper: 1089..."
3,2021:10:12:14:35:33.188219,"X: 161401, Y: 182500, Z: 187000, gripper: 1089..."
4,2021:10:12:14:35:33.536488,"X: 162027, Y: 182500, Z: 187000, gripper: 1089..."
...,...,...
11098,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11099,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11100,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11101,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."


The only useful robot data is postion data.

In [109]:
# Encoding timestamps

# Convert to datetime
df['Timestamp'] = df['Timestamp'].astype(str).str.strip('"').str.strip("'").str.strip()

# Clean and normalize timestamp format
df['Timestamp'] = (
    df['Timestamp']
    .astype(str)
    .str.strip('"')
    .str.strip("'")
    .str.strip()
    .str.replace(":", "-", 2)     # Replace first two colons (Y:M:D → Y-M-D)
    .str.replace(":", "T", 1)     # Replace next colon (between day and hour) with T
)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='ISO8601', utc=True)

# Extract components, only use hour or shorter since all done in a single day
df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['second'] = df['Timestamp'].dt.second
df['microsecond'] = df['Timestamp'].dt.microsecond

df.drop(columns=['Timestamp'], inplace=True)

df['time'] = (
    df['hour'] * 3600 + df['minute'] * 60 + df['second'] + df['microsecond'] / 1_000_000
)
df = df.sort_values('time')
df['time'] =  df['time'] - df['time'].iloc[0]

df.drop(columns=['hour', 'minute', 'second', 'microsecond'], inplace=True)

df.reset_index(drop=True, inplace=True)

df

,Arguments,time
0,"X: -186053, Y: -13900, Z: 250000, gripper: 150...",0.000000
1,"X: -181287, Y: -4773, Z: 250000, gripper: 1506...",0.846160
2,"X: -174021, Y: 2766, Z: 250000, gripper: 15066...",1.290711
3,"X: -165193, Y: 8004, Z: 250000, gripper: 15066...",1.717405
4,"X: -156365, Y: 13004, Z: 250000, gripper: 1506...",2.163429
...,...,...
11098,"X: -154900, Y: 212000, Z: 250000, gripper: 150...",16532.287049
11099,"X: -144900, Y: 212714, Z: 250000, gripper: 150...",16532.646601
11100,"X: -134900, Y: 213429, Z: 250000, gripper: 150...",16533.064527
11101,"X: -124900, Y: 214619, Z: 250000, gripper: 150...",16533.480524


In [110]:
# Extract X, Y, Z values using regex
df[['x', 'y', 'z']] = df['Arguments'].str.extract(
    r'X:\s*(-?\d+),\s*Y:\s*(-?\d+),\s*Z:\s*(-?\d+)'
).astype(int)

df.drop(columns=['Arguments'], inplace=True)

df

,time,x,y,z
0,0.000000,-186053,-13900,250000
1,0.846160,-181287,-4773,250000
2,1.290711,-174021,2766,250000
3,1.717405,-165193,8004,250000
4,2.163429,-156365,13004,250000
...,...,...,...,...
11098,16532.287049,-154900,212000,250000
11099,16532.646601,-144900,212714,250000
11100,16533.064527,-134900,213429,250000
11101,16533.480524,-124900,214619,250000


In [111]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def time_series_plots(df, error=None):
    # Define axis list and colors
    feature_type_lst = ["x", "y", "z"]
    colors = px.colors.qualitative.Dark24[:3]

    # Create 3 subplots for X, Y, Z
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=("X Position", "Y Position", "Z Position"),
        vertical_spacing=0.08
    )

    # Loop over X, Y, Z
    for idx, feature_type in enumerate(feature_type_lst, start=1):
        if feature_type in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[feature_type],
                    name=feature_type.upper(),
                    mode='lines',
                    line=dict(color=colors[idx - 1], width=2)
                ),
                row=idx, col=1
            )

    # Update axes and layout
    fig.update_xaxes(title_text="Time (s)", row=3, col=1, rangeslider_visible=True)
    for i, feature_type in enumerate(feature_type_lst, start=1):
        fig.update_yaxes(title_text=f"{feature_type} (pos)", row=i, col=1)

    fig.update_layout(
        height=900,
        title_text="Position Time Series (X, Y, Z)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

# Example usage:
fig = time_series_plots(df)
fig.show()


In [112]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def report_time_gaps(df, time_col='time', gap_threshold=2.0):
    """
    Prints the size of time gaps exceeding a threshold and reports the longest contiguous segment.

    Parameters:
        df : pd.DataFrame
            Must contain a numeric or datetime time column.
        time_col : str
            Name of the time column.
        gap_threshold : float
            Threshold (in seconds) for gap detection.
    """
    df = df.copy()

    # Convert to numeric seconds if datetime
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        df[time_col] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()

    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)

    # Compute time differences
    time_vals = df[time_col].to_numpy()
    gaps = np.diff(time_vals)

    print(f"\nChecking for gaps > {gap_threshold} seconds...\n")

    gap_indices = np.where(gaps > gap_threshold)[0]
    gap_sizes = gaps[gaps > gap_threshold]

    if len(gap_sizes) == 0:
        print("No gaps exceeding threshold found.")
        print(f"Longest contiguous segment: {time_vals[0]:.3f}s to {time_vals[-1]:.3f}s "
              f"({time_vals[-1] - time_vals[0]:.3f} seconds long)")
        return

    # Print all gaps
    for gap in gap_sizes:
        print(f"Gap size: {gap:.3f} seconds")

    print(f"\nTotal gaps found: {len(gap_sizes)}")

    # Find segment boundaries between gaps
    segment_starts = np.concatenate(([0], gap_indices + 1))
    segment_ends = np.concatenate((gap_indices + 1, [len(time_vals)]))

    # Compute segment lengths
    segment_lengths = [time_vals[end - 1] - time_vals[start] for start, end in zip(segment_starts, segment_ends)]

    # Identify the longest segment
    longest_idx = np.argmax(segment_lengths)
    start_idx, end_idx = segment_starts[longest_idx], segment_ends[longest_idx]
    longest_duration = segment_lengths[longest_idx]

    print(f"\nLongest contiguous segment: {time_vals[start_idx]:.3f}s to {time_vals[end_idx - 1]:.3f}s "
          f"({longest_duration:.3f} seconds long)")

report_time_gaps(df, time_col='time', gap_threshold=5.0)


Checking for gaps > 5.0 seconds...

Gap size: 2498.999 seconds
Gap size: 744.197 seconds
Gap size: 869.179 seconds
Gap size: 1025.658 seconds
Gap size: 38.464 seconds
Gap size: 55.822 seconds
Gap size: 8.641 seconds
Gap size: 73.150 seconds
Gap size: 279.219 seconds
Gap size: 228.868 seconds
Gap size: 410.528 seconds
Gap size: 145.404 seconds
Gap size: 6.021 seconds
Gap size: 29.699 seconds
Gap size: 36.002 seconds
Gap size: 34.933 seconds
Gap size: 11.518 seconds
Gap size: 6.285 seconds
Gap size: 9.866 seconds
Gap size: 18.137 seconds
Gap size: 5.060 seconds
Gap size: 8.882 seconds
Gap size: 253.200 seconds
Gap size: 10.527 seconds
Gap size: 8.562 seconds
Gap size: 13.703 seconds
Gap size: 15.224 seconds
Gap size: 10.702 seconds
Gap size: 8.085 seconds
Gap size: 5.439 seconds
Gap size: 341.775 seconds
Gap size: 165.650 seconds
Gap size: 57.444 seconds
Gap size: 164.054 seconds
Gap size: 143.785 seconds
Gap size: 258.474 seconds
Gap size: 132.480 seconds
Gap size: 51.779 seconds
Gap s

In [113]:
import plotly.graph_objects as go


# Create a 3D scatter (or line) plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df['x'],
            y=df['y'],
            z=df['z'],
            mode='lines+markers',  # 'lines', 'markers', or 'lines+markers'
            line=dict(color='royalblue', width=4),
            marker=dict(size=3, color='orange'),
            hovertemplate=
            'Time: %{customdata:.2f}s<extra></extra>',
            customdata=df['time'])
    ]
)

# Add layout details
fig.update_layout(
    title='3D Position Trajectory',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    height=700
)

fig.show()


Looking at the robot's position over the course of the recovered data, there are large gaps where the robot appears to teleport through space. Some of these are simply missing readings where the robot went AWOL for some time, even up to 2600 seconds, nearly 45 minutes. What is was doing during that time, no one can say. Perhaps it was between runs at that time. 

There are also gaps in space that have only a short gap in time, too short for it to actually travel to that point. This suggesets misordering in the data or else a position encoder error.

In [114]:
import plotly.graph_objects as go

min_points = 83
max_points = 335  # Limit number of points for clarity
# Create a 3D scatter (or line) plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df['x'][min_points:max_points],
            y=df['y'][min_points:max_points],
            z=df['z'][min_points:max_points],
            mode='lines+markers',  # 'lines', 'markers', or 'lines+markers'
            line=dict(color='royalblue', width=4),
            marker=dict(size=3, color='orange'),
            hovertemplate=
            'Time: %{customdata:.2f}s<extra></extra>',
            customdata=df['time'][min_points:max_points]        )
    ]
)

# Add layout details
fig.update_layout(
    title='3D Position Trajectory',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    height=700
)

fig.show()


For our purposes we will focus on a short 4-minute interval that contains no major time or space disconinuities.

In [115]:
def detailed_summary(df):
    summary = pd.DataFrame(index=df.columns)

    summary["Data Type"] = df.dtypes
    summary["Unique Values"] = df.nunique()

    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        summary.loc[col, "Mean"] = df[col].mean()
        summary.loc[col, "Median"] = df[col].median()
        summary.loc[col, "Std Dev"] = df[col].std()
        summary.loc[col, "Variance"] = df[col].var()
        summary.loc[col, "Min"] = df[col].min()
        summary.loc[col, "Max"] = df[col].max()
        summary.loc[col, "Skewness"] = df[col].skew()
        summary.loc[col, "Kurtosis"] = df[col].kurt()
        summary.loc[col, "25%"] = df[col].quantile(0.25)
        summary.loc[col, "75%"] = df[col].quantile(0.75)
        summary.loc[col, "IQR"] = summary.loc[col, "75%"] - summary.loc[col, "25%"]

    # Round numeric columns neatly
    summary = summary.round(3)
    return summary

detailed_summary(df)

,Data Type,Unique Values,Mean,Median,Std Dev,Variance,Min,Max,Skewness,Kurtosis,25%,75%,IQR
time,float64,8686,10094.375,10279.16,2782.776,7.743841e+06,0.0,16533.898,-0.241,-0.849,7957.466,12677.564,4720.098
x,int64,3895,13425.357,-19032.00,179127.267,3.208658e+10,-332647.0,349992.000,0.194,-1.355,-141600.000,205502.000,347102.000
y,int64,1873,134558.232,185704.00,151061.688,2.281963e+10,-127777.0,343144.000,-0.574,-0.981,-13700.000,263982.000,277682.000
z,int64,1322,264687.802,293890.00,59036.444,3.485302e+09,98780.0,319091.000,-1.185,-0.019,230070.000,308244.000,78174.000


In [128]:
def histogram_plots(df_cobots, feature_type_lst=["Current", "Speed", "Temperature"], unit=["A", "m/s", "Degrees C"]):

    colors = px.colors.qualitative.Dark24

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=[f'{feature} Distribution' for feature in feature_type_lst],
        horizontal_spacing=0.1
    )

    # Loop through each feature type
    for feat_idx, (feature_type, unit_label) in enumerate(zip(feature_type_lst, unit)):
        row = feat_idx + 1  
        
        fig.add_trace(
            go.Histogram(
                x=df_cobots[feature_type],
                marker=dict(color=colors[feat_idx]),
                opacity=0.7,
            ),
            row=row, col=1
        )
        
        fig.update_xaxes(title_text=f"{feature_type} ({unit_label})", row=row, col=1)

    fig.update_yaxes(title_text="Count", row=1, col=1)

    fig.update_layout(
        height=1000,
        barmode='overlay',  
        showlegend=False,

    )
    return fig

feature_type_lst = ["x", "y", "z"]
unit_lst = ["pos", "pos", "pos"]
fig = histogram_plots(df, feature_type_lst=feature_type_lst, unit=unit_lst)
fig.show()

In [126]:
import plotly.graph_objects as go
import numpy as np


 # Calculate correlation
df_corr = df.drop(columns=['time']).corr().round(2)

# Mask upper triangle
mask = np.zeros_like(df_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# Apply mask and drop empty rows/cols
df_corr_viz = df_corr.mask(mask).dropna(how='all').dropna(axis='columns', how='all')

# Create text array with blanks instead of nan
text_values = df_corr_viz.values.astype(str)
text_values[text_values == 'nan'] = ''

# Add heatmap to subplot
fig =  go.Heatmap(
        z=df_corr_viz.values,
        x=df_corr_viz.columns,
        y=df_corr_viz.index,
        colorscale='Viridis',
        zmid=0,
        text=text_values,
        texttemplate='%{text}',
        textfont={"size": 8},
    )

fig = go.Figure(data=fig)
fig.update_layout(
    height=400,
    width=425,
    title_text="Correlation Analysis",
    showlegend=False
)
fig.show()